In [0]:
# =============================================================================
# NOTEBOOK  : silver_pos_transactions_stream
# PURPOSE   : bronze.pos_transactions (stream rows) → silver.pos_transactions
# LAYER     : Silver (clean, type-cast, dedup, merge)
# FREQUENCY : Continuous micro-batch (JOB A — runs 24/7)
# TRIGGER   : availableNow (dev) / no trigger (prod)
#
# MERGE & DEDUP LOGIC:
#   Source  : bronze.pos_transactions WHERE _source = 'pos_realtime_stream'
#   Checkpoint tracks bronze Delta log offset — not row-level filters
#   Stream and batch silver have SEPARATE checkpoints — no interference
#
#   Within micro-batch dedup:
#     Assumption: duplicate records in same micro-batch are EXACTLY identical
#     → deduplicate on transaction_id BEFORE merge via dropDuplicates
#     → keep any one — all copies are identical
#     WHY before merge: Delta MERGE does not deduplicate source df internally
#     Two rows with same key in source = two inserts regardless of merge logic
#
#   MERGE cases:
#     Case 1: transaction_id NOT in silver → INSERT
#     Case 2: transaction_id IN silver     → IGNORE (no whenMatchedUpdate)
#
#   Stream silver NEVER updates or deletes silver records
#   Updates and deletes are exclusively batch silver's responsibility
# =============================================================================
 
# ── Imports & Config ──────────────────────────────────────────────────
import sys, json
sys.path.insert(0, "/Workspace/Shared/mini_project_a3t7")
 
from utils.audit import start_audit, end_audit
from utils.schema_registry import SILVER_POS_TRANSACTIONS_SCHEMA
 
from pyspark.sql.functions import (
    current_timestamp, lit, col,
    to_date, to_timestamp
)
from pyspark.sql.types import DecimalType
from delta.tables import DeltaTable
 
with open("/Workspace/Shared/mini_project_a3t7/config/config.json") as f:
    cfg = json.load(f)
 
SOURCE_TABLE = cfg["tables"]["bronze_pos_transactions"]
TARGET_TABLE = cfg["tables"]["silver_pos_transactions"]
CHECKPOINT   = cfg["checkpoint_paths"]["silver_pos_transactions_stream"]
PIPELINE     = "silver_pos_transactions_stream"

In [0]:
# ── foreachBatch function + Stream ────────────────────────────────────
 
def process_microbatch(micro_batch_df, microbatch_id):
    """
    Called by Spark for every micro-batch of bronze rows.
 
    FILTER   : Only stream rows (_source = pos_realtime_stream)
    DEDUP    : dropDuplicates on transaction_id before MERGE
               Assumption: duplicates are exactly identical records
    MERGE    : Case 1 → INSERT, Case 2 → IGNORE
    """
    
    # ── executor-side imports ─────────────────────────────────
    import sys
    sys.path.insert(0, "/Workspace/Shared/mini_project_a3t7")
    from utils.audit import start_audit, end_audit
    from utils.schema_registry import SILVER_POS_TRANSACTIONS_SCHEMA

    # ── FILTER — stream rows only ─────────────────────────────────────────────
    micro_batch_df = micro_batch_df.filter(
        col("_source") == "pos_realtime_stream"
    )
 
    if micro_batch_df.isEmpty():
        return
 
    run_id = start_audit(spark, PIPELINE, TARGET_TABLE, SOURCE_TABLE)
 
    try:
        rows_read = micro_batch_df.count()
 
        # ── TRANSFORM ─────────────────────────────────────────────────────────
        df = (
            micro_batch_df

            # 1. Cast timestamp ISO string → TimestampType
            #    Source format: "2023-08-19T22:26:11Z"
            #    Renamed to transaction_ts — avoids Spark reserved word 'timestamp'
            .withColumn("transaction_ts",  to_timestamp(col("timestamp"), "yyyy-MM-dd'T'HH:mm:ss'Z'"))

            # 2. Money columns → Decimal for precision
            #    Double has floating point imprecision — critical for financial data
            .withColumn("unit_price",      col("unit_price").cast(DecimalType(10, 2)))
            .withColumn("total_amount",    col("total_amount").cast(DecimalType(10, 2)))

            # 3. transaction_date derived from event time (transaction_ts)
            #    NOT from ingested_at — silver partitions by when event happened
            .withColumn("transaction_date",to_date(to_timestamp(col("timestamp"), "yyyy-MM-dd'T'HH:mm:ss'Z'")).cast("string"))

            # 4. Silver audit columns
            .withColumn("processed_at",    current_timestamp())
            .withColumn("pipeline_run_id", lit(run_id))
            .withColumn("source_system",   lit("pos_realtime_stream"))

            # 5. Enforce silver schema — drops bronze-only cols in one line
            #    (_source, source_file, ingested_date, etc.)
            .select([f.name for f in SILVER_POS_TRANSACTIONS_SCHEMA.fields])
        )
 
        # ── DEDUP WITHIN MICRO-BATCH ────────────────────────────────
        # Must happen BEFORE merge
        # Delta MERGE does not deduplicate source df internally
        # Two rows with same transaction_id in source = two inserts
        df = df.dropDuplicates(["transaction_id"])
 
        rows_after_dedup = df.count()
        if rows_read != rows_after_dedup:
            print(f"[DEDUP] dropped {rows_read - rows_after_dedup} "
                  f"duplicate stream rows in microbatch_id={microbatch_id}")
 
        # ── MERGE INTO SILVER ──────────────────────────────────────────────────
        # Case 1: not in silver → INSERT
        # Case 2: in silver     → no whenMatchedUpdate → IGNORE
        (
            DeltaTable.forName(spark, TARGET_TABLE).alias("t")
            .merge(
                df.alias("s"),
                "t.transaction_id = s.transaction_id"
            )
            .whenNotMatchedInsertAll()
            .execute()
        )
 
        metrics = (
            spark.sql(f"DESCRIBE HISTORY {TARGET_TABLE} LIMIT 1")
            .select("operationMetrics")
            .collect()[0][0]
        )
        rows_inserted = int(metrics.get("numTargetRowsInserted", 0))
 
        end_audit(spark, run_id, "SUCCESS",
                  rows_read=rows_read,
                  rows_written=rows_inserted,
                  "microbatch_id": str(microbatch_id)
        )
 
    except Exception as e:
        end_audit(spark, run_id, "FAILED", error=str(e))
        raise
 
 
# ── READ + WRITE ──────────────────────────────────────────────────────────────
bronze_stream = (
    spark.readStream
    .format("delta")
    .option("ignoreChanges", "true")
    .table(SOURCE_TABLE)
)
 
query = (
    bronze_stream.writeStream
    .foreachBatch(process_microbatch)
    .option("checkpointLocation", CHECKPOINT)
    .trigger(availableNow=True)   # ← remove for prod continuous mode
    .start()
)
 
query.awaitTermination()

In [0]:
# ── Verify last run ───────────────────────────────────────────────────
# Run this cell manually after the stream completes to verify the run

# ── 1. Latest audit entry for this pipeline ───────────────────────────────────
print("=" * 60)
print("AUDIT LOG — Last 3 runs")
print("=" * 60)
(
    spark.read.table("azure3_team7_project.platform.pipeline_audit")
    .filter(col("pipeline_name") == PIPELINE)
    .orderBy(col("start_time").desc())
    .limit(3)
    .select(
        "run_id", "status", "rows_read",
        "rows_written", "rows_rejected",
        "start_time", "end_time", "error_message"
    )
    .show(truncate=False)
)

# ── 2. Row counts in silver by source_system ──────────────────────────────────
print("=" * 60)
print("SILVER TABLE — Row counts by source_system")
print("=" * 60)
(
    spark.read.table(TARGET_TABLE)
    .groupBy("source_system")
    .count()
    .show(truncate=False)
)

# ── 3. Row counts by transaction_date ─────────────────────────────────────────
print("=" * 60)
print("SILVER TABLE — Row counts by transaction_date (last 5 dates)")
print("=" * 60)
(
    spark.read.table(TARGET_TABLE)
    .groupBy("transaction_date")
    .count()
    .orderBy(col("transaction_date").desc())
    .limit(5)
    .show(truncate=False)
)

# ── 4. Sample rows from latest pipeline_run_id ────────────────────────────────
print("=" * 60)
print("SILVER TABLE — Sample rows from latest run")
print("=" * 60)
latest_run_id = (
    spark.read.table("azure3_team7_project.platform.pipeline_audit")
    .filter(col("pipeline_name") == PIPELINE)
    .filter(col("status") == "SUCCESS")
    .orderBy(col("start_time").desc())
    .limit(1)
    .select("run_id")
    .collect()[0][0]
)
print(f"Latest successful run_id: {latest_run_id}")
(
    spark.read.table(TARGET_TABLE)
    .filter(col("pipeline_run_id") == latest_run_id)
    .limit(5)
    .select(
        "transaction_id", "store_id", "transaction_ts",
        "total_amount", "source_system", "transaction_date"
    )
    .show(truncate=False)
)

# ── 5. Delta history — last 3 operations on silver table ──────────────────────
print("=" * 60)
print("DELTA HISTORY — Last 3 operations on silver table")
print("=" * 60)
(
    spark.sql(f"DESCRIBE HISTORY {TARGET_TABLE} LIMIT 3")
    .select(
        "version", "timestamp", "operation",
        "operationMetrics", "userName"
    )
    .show(truncate=False)
)